# Sample Adjustment Check

Use this notebook to compare available BTC CSV samples, inspect the detected schema, and preview the cleaned FI-2010-style output before training.

In [ ]:
from pathlib import Path

import pandas as pd

from clean_tardis_lob_dataset import (
    DEFAULT_CHUNK_SIZE,
    DEFAULT_DEPTH_LEVELS,
    add_bucket_columns,
    clean_tardis_csv,
    detect_input_format,
    normalize_book_columns,
    profile_timestamps,
)

data_dir = Path.cwd() / 'data' / 'Tardis_sample_data'
candidate_files = sorted(data_dir.glob('*.csv'))
if not candidate_files:
    raise FileNotFoundError(f'No CSV files found in {data_dir}')

selected_path = data_dir / 'BTC20230120.csv'
if not selected_path.exists():
    selected_path = candidate_files[-1]

print('available files:')
for path in candidate_files:
    print(f'  - {path.name}')
print(f'\nselected file: {selected_path.name}')

In [ ]:
comparison_rows = []
for path in candidate_files:
    input_format = detect_input_format(path, depth_levels=DEFAULT_DEPTH_LEVELS)
    profile = profile_timestamps(path, chunk_size=DEFAULT_CHUNK_SIZE, input_format=input_format)
    comparison_rows.append(
        {
            'file': path.name,
            'format': input_format.name,
            'rows_raw': profile.total_rows,
            'rows_unique_ms': profile.unique_ms_rows,
            'rows_clean_1s': profile.seconds_covered,
            'duplicate_ms_rows': profile.duplicate_ms_rows,
            'avg_rows_per_second': profile.avg_rows_per_second,
            'max_rows_per_second': profile.max_rows_per_second,
            'recommended_frequency': profile.recommended_frequency,
            'est_sequences_len10': max(profile.seconds_covered - 10 + 1, 0),
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values('rows_clean_1s', ascending=False).reset_index(drop=True)
comparison

In [ ]:
raw_preview = pd.read_csv(selected_path, nrows=5)
raw_preview

In [ ]:
selected_format = detect_input_format(selected_path, depth_levels=DEFAULT_DEPTH_LEVELS)
preview_chunk = pd.read_csv(selected_path, usecols=selected_format.usecols, nrows=5)
normalized_preview = normalize_book_columns(preview_chunk, depth_levels=DEFAULT_DEPTH_LEVELS, input_format=selected_format)
normalized_preview = add_bucket_columns(normalized_preview, freq='1s')
normalized_preview[['exchange', 'symbol', 'timestamp_utc', 'bid_price_1', 'ask_price_1', 'mid_price', 'spread']]


In [ ]:
selected_clean = clean_tardis_csv(
    input_path=selected_path,
    depth_levels=DEFAULT_DEPTH_LEVELS,
    freq='1s',
    chunk_size=DEFAULT_CHUNK_SIZE,
    input_format=selected_format,
)
print(selected_clean.shape)
selected_clean[['timestamp_utc', 'bid_price_1', 'ask_price_1', 'mid_price', 'spread']].head()